In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split

class GeneradorSegmentacionTFG(Sequence):
    def __init__(self, carpeta_parches, lista_archivos, batch_size=64, shuffle=True):
        self.carpeta = carpeta_parches
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.archivos_X = lista_archivos
        self.num_clases = 20
        self.indices = np.arange(len(self.archivos_X))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.archivos_X) / self.batch_size))

    def __getitem__(self, index):
        indices_batch = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_X = []
        batch_Y = []
        
        for idx in indices_batch:
            nom_X = self.archivos_X[idx]
            nom_Y = nom_X.replace('_X.npy', '_Y.npy')
            
            parche_X = np.load(os.path.join(self.carpeta, nom_X)).astype(np.float32)
            parche_Y = np.load(os.path.join(self.carpeta, nom_Y)).astype(np.int32)
            
            if parche_X.shape[0] == 4:
                parche_X = np.transpose(parche_X, (1, 2, 0))
            
            parche_X = parche_X[:48, :48, :]
            parche_Y = parche_Y[:48, :48]
            
            parche_Y = np.where((parche_Y >= 0) & (parche_Y <= 19), parche_Y, 19)
            
            mask_one_hot = tf.one_hot(parche_Y, depth=self.num_clases, dtype=tf.float32).numpy()
            
            batch_X.append(parche_X)
            batch_Y.append(mask_one_hot)
            
        if len(batch_X) == 0:
            return (np.zeros((1, 48, 48, 4), dtype=np.float32), 
                    np.zeros((1, 48, 48, self.num_clases), dtype=np.float32))
            
        return np.array(batch_X), np.array(batch_Y)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# --- 1. FILTRADO INTELIGENTE PREVIO DEL DATASET ---
carpeta_dataset = r"D:\Jorge\dataset_parches_50x50\dataset_parches_50x50"

print("Paso 1: Escaneando nombres de archivos en el disco...", flush=True)
todos_los_X_total = [f for f in os.listdir(carpeta_dataset) if f.endswith('_X.npy')]

np.random.seed(4215) 
np.random.shuffle(todos_los_X_total)

print("Paso 2: Buscando parches útiles reales hasta completar tu cupo...", flush=True)
archivos_prueba = []
cupo_deseado = 150000  

for nom_X in todos_los_X_total:
    if len(archivos_prueba) >= cupo_deseado:
        break 
        
    nom_Y = nom_X.replace('_X.npy', '_Y.npy')
    try:
        parche_y = np.load(os.path.join(carpeta_dataset, nom_Y))
        conteos = np.bincount(parche_y.ravel(), minlength=20)
        
        if np.argmax(conteos[:20]) != 19:
            archivos_prueba.append(nom_X)
    except Exception:
        continue

print(f"¡Listo! Dataset seleccionado con {len(archivos_prueba):,} parches de alta calidad.")

archivos_train, archivos_val = train_test_split(archivos_prueba, test_size=0.2, random_state=42)

generador_train = GeneradorSegmentacionTFG(carpeta_dataset, archivos_train, batch_size=32, shuffle=True)
generador_val = GeneradorSegmentacionTFG(carpeta_dataset, archivos_val, batch_size=32, shuffle=False)

# --- 2. CONFIGURACIÓN DE LA ARQUITECTURA DE LA U-NET REAL ---
def build_unet_autoencoder(input_shape, num_classes):
    entradas = layers.Input(shape=input_shape)
    init = 'he_normal'
    reg = tf.keras.regularizers.l2(1e-6)
    
    x = layers.Normalization(axis=-1)(entradas)
    
    # --- ENCODER ---
    # Bloque 1 (48x48)
    c1 = layers.Conv2D(32, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(x)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Activation('gelu')(c1)
    c1 = layers.Conv2D(32, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(c1)
    c1 = layers.BatchNormalization()(c1)
    skip1 = layers.Activation('gelu')(c1) # Conexión de salto 1
    p1 = layers.MaxPooling2D((2, 2))(skip1) # Reduce a 24x24
    
    # Bloque 2 (24x24)
    c2 = layers.Conv2D(64, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Activation('gelu')(c2)
    c2 = layers.Conv2D(64, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(c2)
    c2 = layers.BatchNormalization()(c2)
    skip2 = layers.Activation('gelu')(c2) # Conexión de salto 2
    p2 = layers.MaxPooling2D((2, 2))(skip2) # Reduce a 12x12
    
    # --- BOTTLENECK ---
    bottleneck = layers.Conv2D(128, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(p2)
    bottleneck = layers.BatchNormalization()(bottleneck)
    bottleneck = layers.Activation('gelu')(bottleneck)
    bottleneck = layers.Conv2D(128, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(bottleneck)
    bottleneck = layers.BatchNormalization()(bottleneck)
    bottleneck = layers.Activation('gelu')(bottleneck)
    
    # --- DECODER ---
    # Bloque 3 (Sube a 24x24)
    u3 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same', kernel_initializer=init, kernel_regularizer=reg)(bottleneck)
    merge3 = layers.concatenate([u3, skip2]) # Concatenamos simétricamente con skip2
    c3 = layers.Conv2D(64, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(merge3)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation('gelu')(c3)
    c3 = layers.Conv2D(64, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(c3)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation('gelu')(c3)
    
    # Bloque 4 (Sube a 48x48)
    u4 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same', kernel_initializer=init, kernel_regularizer=reg)(c3)
    merge4 = layers.concatenate([u4, skip1]) # Concatenamos simétricamente con skip1
    c4 = layers.Conv2D(32, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(merge4)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Activation('gelu')(c4)
    c4 = layers.Conv2D(32, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(c4)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Activation('gelu')(c4)
    
    # Corrección sintáctica del cálculo del bias_init que causaba el fallo
    prior_prob = 0.01
    bias_init = tf.constant_initializer(-np.log((1.0 - prior_prob) / prior_prob))

    salidas = layers.Conv2D(num_classes, (1, 1), activation='sigmoid', kernel_initializer='glorot_uniform', bias_initializer=bias_init)(c4)
    
    return models.Model(inputs=entradas, outputs=salidas)

model = build_unet_autoencoder((48, 48, 4), 20)

# --- 3. PESOS DE TU FUNCIÓN DE PÉRDIDA ---
PESOS_REALES_MESETA = tf.constant([
    23.4,  # Clase 0
    69.1,  # Clase 1
    1.0,   # Clase 2
    4.0,   # Clase 3
    19.2,  # Clase 4
    7.6,   # Clase 5
    11.0,  # Clase 6
    8.3,   # Clase 7
    1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, # Clases 8 a 18
    0.0    # Clase 19
], dtype=tf.float32)

def loss_bce_pesada(y_true, y_pred):
    smooth = 1e-5
    y_pred = tf.clip_by_value(y_pred, smooth, 1.0 - smooth)
    
    bce_por_canal = - (y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    bce_pesada = bce_por_canal * PESOS_REALES_MESETA
    
    return tf.reduce_mean(tf.reduce_sum(bce_pesada, axis=-1))

# --- 4. COMPILACIÓN ORIGINAL ---
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3, clipvalue=0.5),
    loss=loss_bce_pesada, 
    metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')] 
)

mis_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1)
]

history = model.fit(
    generador_train,
    validation_data=generador_val,
    epochs=40,
    callbacks=mis_callbacks
)

Paso 1: Escaneando nombres de archivos en el disco...
Paso 2: Buscando parches útiles reales hasta completar tu cupo...
¡Listo! Dataset seleccionado con 150,000 parches de alta calidad.
Epoch 1/40
3750/3750 [==============================] - 872s 231ms/step - loss: 11.6307 - accuracy: 0.6115 - val_loss: 12.4077 - val_accuracy: 0.5827 - lr: 0.0010
Epoch 2/40
3750/3750 [==============================] - 364s 97ms/step - loss: 10.1857 - accuracy: 0.6670 - val_loss: 14.4409 - val_accuracy: 0.5784 - lr: 0.0010
Epoch 3/40
3750/3750 [==============================] - 372s 99ms/step - loss: 9.7346 - accuracy: 0.6828 - val_loss: 11.7599 - val_accuracy: 0.5135 - lr: 0.0010
Epoch 4/40
3750/3750 [==============================] - 425s 113ms/step - loss: 9.4055 - accuracy: 0.6922 - val_loss: 11.1582 - val_accuracy: 0.6676 - lr: 0.0010
Epoch 5/40
3750/3750 [==============================] - 449s 120ms/step - loss: 9.2310 - accuracy: 0.6964 - val_loss: 9.6748 - val_accuracy: 0.6699 - lr: 0.0010
Epoch

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split

class GeneradorSegmentacionTFG(Sequence):
    def __init__(self, carpeta_parches, lista_archivos, batch_size=64, shuffle=True):
        self.carpeta = carpeta_parches
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.archivos_X = lista_archivos
        # Cambiado a 20 porque la clase '19' requiere que el vector tenga longitud 20 (índices 0 al 19)
        self.num_clases = 20
        self.indices = np.arange(len(self.archivos_X))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.archivos_X) / self.batch_size))

    def __getitem__(self, index):
        indices_batch = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_X = []
        batch_Y = []
        
        for idx in indices_batch:
            nom_X = self.archivos_X[idx]
            nom_Y = nom_X.replace('_X.npy', '_Y.npy')
            
            parche_X = np.load(os.path.join(self.carpeta, nom_X)).astype(np.float32)
            parche_Y = np.load(os.path.join(self.carpeta, nom_Y)).astype(np.int32)
            
            if parche_X.shape[0] == 4:
                parche_X = np.transpose(parche_X, (1, 2, 0))
            
            parche_X = parche_X[:48, :48, :]
            parche_Y = parche_Y[:48, :48]
            
            # Si se coló algún código corrupto fuera del rango [0, 19], lo forzamos a ser 19
            parche_Y = np.where((parche_Y >= 0) & (parche_Y <= 19), parche_Y, 19)
            
            mask_one_hot = tf.one_hot(parche_Y, depth=self.num_clases, dtype=tf.float32).numpy()
            
            batch_X.append(parche_X)
            batch_Y.append(mask_one_hot)
            
        if len(batch_X) == 0:
            return (np.zeros((1, 48, 48, 4), dtype=np.float32), 
                    np.zeros((1, 48, 48, self.num_clases), dtype=np.float32))
            
        return np.array(batch_X), np.array(batch_Y)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# --- 1. FILTRADO INTELIGENTE PREVIO DEL DATASET (EVITAR PARCHES VACÍOS) ---
carpeta_dataset = r"D:\Jorge\dataset_parches_50x50\dataset_parches_50x50"

print("Paso 1: Escaneando nombres de archivos en el disco...", flush=True)
todos_los_X_total = [f for f in os.listdir(carpeta_dataset) if f.endswith('_X.npy')]

# Mezclamos todos los nombres aleatoriamente
np.random.seed(4215) 
np.random.shuffle(todos_los_X_total)

print("Paso 2: Buscando parches útiles reales hasta completar tu cupo...", flush=True)
archivos_prueba = []
cupo_deseado = 150000  

for nom_X in todos_los_X_total:
    if len(archivos_prueba) >= cupo_deseado:
        break 
        
    nom_Y = nom_X.replace('_X.npy', '_Y.npy')
    try:
        # Cargamos solo si el archivo existe
        parche_y = np.load(os.path.join(carpeta_dataset, nom_Y))
        conteos = np.bincount(parche_y.ravel(), minlength=20)
        
        # Si NO está dominado por el vacío (19), directo a la lista de entrenamiento
        if np.argmax(conteos[:20]) != 19:
            archivos_prueba.append(nom_X)
    except Exception:
        continue

print(f"¡Listo! Dataset seleccionado con {len(archivos_prueba):,} parches de alta calidad.")

archivos_train, archivos_val = train_test_split(archivos_prueba, test_size=0.2, random_state=42)

generador_train = GeneradorSegmentacionTFG(carpeta_dataset, archivos_train, batch_size=32, shuffle=True)
generador_val = GeneradorSegmentacionTFG(carpeta_dataset, archivos_val, batch_size=32, shuffle=False)

# --- 2. CONFIGURACIÓN DE LA ARQUITECTURA DE LA U-NET REAL ---
def build_unet_autoencoder(input_shape, num_classes):
    entradas = layers.Input(shape=input_shape)
    init = 'he_normal'  # Cambiado a 'he_normal' por rendimiento óptimo con GELU
    reg = tf.keras.regularizers.l2(1e-6)
    
    x = layers.Normalization(axis=-1)(entradas)
    
    # --- ENCODER ---
    # Bloque 1 (Entrada: 48x48 -> Salida Espacial: 48x48)
    c1 = layers.Conv2D(32, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(x)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Activation('gelu')(c1)
    c1 = layers.Conv2D(32, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(c1)
    c1 = layers.BatchNormalization()(c1)
    skip1 = layers.Activation('gelu')(c1)   # <-- Conexión de salto 1 pura
    p1 = layers.MaxPooling2D((2, 2))(skip1)  # Reduce a 24x24
    
    # Bloque 2 (Entrada: 24x24 -> Salida Espacial: 24x24)
    c2 = layers.Conv2D(64, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Activation('gelu')(c2)
    c2 = layers.Conv2D(64, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(c2)
    c2 = layers.BatchNormalization()(c2)
    skip2 = layers.Activation('gelu')(c2)   # <-- Conexión de salto 2 pura
    p2 = layers.MaxPooling2D((2, 2))(skip2)  # Reduce a 12x12
    
    # --- BOTTLENECK ---
    bottleneck = layers.Conv2D(128, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(p2)
    bottleneck = layers.BatchNormalization()(bottleneck)
    bottleneck = layers.Activation('gelu')(bottleneck)
    bottleneck = layers.Conv2D(128, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(bottleneck)
    bottleneck = layers.BatchNormalization()(bottleneck)
    bottleneck = layers.Activation('gelu')(bottleneck)
    
    # --- DECODER ---
    # Bloque 3: Sube de 12x12 a 24x24
    u3 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same', kernel_initializer=init, kernel_regularizer=reg)(bottleneck)
    merge3 = layers.concatenate([u3, skip2]) # <-- Concatenación simétrica con skip2 (24x24)
    c3 = layers.Conv2D(64, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(merge3)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation('gelu')(c3)
    c3 = layers.Conv2D(64, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(c3)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation('gelu')(c3)
    
    # Bloque 4: Sube de 24x24 a 48x48
    u4 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same', kernel_initializer=init, kernel_regularizer=reg)(c3)
    merge4 = layers.concatenate([u4, skip1]) # <-- Concatenación simétrica con skip1 (48x48)
    c4 = layers.Conv2D(32, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(merge4)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Activation('gelu')(c4)
    c4 = layers.Conv2D(32, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(c4)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Activation('gelu')(c4)
    
    # Capa final de clasificación por píxel
    salidas = layers.Conv2D(num_classes, (1, 1), activation='softmax', kernel_initializer='glorot_uniform')(c4)
    
    return models.Model(inputs=entradas, outputs=salidas)

# Configuramos la red para manejar el mapa completo de salidas de tamaño 20
model = build_unet_autoencoder((48, 48, 4), 20)

# --- 3. FUNCIÓN DE PÉRDIDA PESADA PERSONALIZADA (IGNORANDO EL CANAL 19) ---
PESOS_REALES_MESETA = tf.constant([
    23.4,  # Clase 0
    69.1,  # Clase 1
    1.0,   # Clase 2 (Base mayoritaria de la meseta central)
    4.0,   # Clase 3
    19.2,  # Clase 4
    7.6,   # Clase 5
    11.0,  # Clase 6
    8.3,   # Clase 7
    1.0,   # Clase 8 (Vacia en tu zona)
    1.0,   # Clase 9 (Vacia en tu zona)
    1.0,   # Clase 10 (Vacia en tu zona)
    1.0,   # Clase 11 (Residual)
    1.0,   # Clase 12 (Vacia en tu zona)
    1.0,   # Clase 13 (Vacia en tu zona)
    1.0,   # Clase 14 (Vacia en tu zona)
    1.0,   # Clase 15 (Vacia en tu zona)
    1.0,   # Clase 16 (Vacia en tu zona)
    1.0,   # Clase 17 (Vacia en tu zona)
    1.0,   # Clase 18 (Vacia en tu zona)
    0.0    # Clase 19 (PESO CERO -> El No Data no penaliza ni modifica gradientes)
], dtype=tf.float32)

def loss_cross_entropy_pesada(y_true, y_pred):
    smooth = 1e-5
    y_pred = tf.clip_by_value(y_pred, smooth, 1.0 - smooth)
    
    # Entropía cruzada píxel a píxel
    cce_por_pixel = -y_true * tf.math.log(y_pred)
    
    # Aplicamos la máscara de pesos (incluyendo el cero para anular la clase 19)
    cce_pesada = cce_por_pixel * PESOS_REALES_MESETA
    
    return tf.reduce_mean(tf.reduce_sum(cce_pesada, axis=-1))

# --- 4. COMPILACIÓN Y ENTRENAMIENTO ---
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3, clipvalue=0.5),
    loss=loss_cross_entropy_pesada, 
    metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')]
)

mis_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1)
]

history = model.fit(
    generador_train,
    validation_data=generador_val,
    epochs=40,
    callbacks=mis_callbacks
)